<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/5_audio_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datasets import load_dataset
from datasets import Audio

minds = load_dataset("PolyAI/minds14", name="en-AU", split="train")

In [3]:
minds

Dataset({
    features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
    num_rows: 654
})

In [10]:
sample1=minds[0]
#采样率
sample1["audio"]["sampling_rate"]

16000

In [11]:
#重新采样
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))

In [12]:
from transformers import pipeline

#第一个任务：audio classification
classifier = pipeline(
    "audio-classification",
    model="anton-l/xtreme_s_xlsr_300m_minds14",
)

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [23]:
#audio相关信息
audio = sample1["audio"]
samples = audio.get_all_samples()
samples

AudioSamples:
  data (shape): torch.Size([1, 124830])
  pts_seconds: 0.0
  duration_seconds: 7.801875
  sample_rate: 16000

In [27]:
#audio的array相关信息
audio_array=audio["array"]

In [28]:
#用classifier进行分类，可以看到pay_bill概率最大
classifier(audio_array)

[{'score': 0.9658821225166321, 'label': 'pay_bill'},
 {'score': 0.0258216243237257, 'label': 'freeze'},
 {'score': 0.003020624630153179, 'label': 'card_issues'},
 {'score': 0.0018971438985317945, 'label': 'abroad'},
 {'score': 0.000823002599645406, 'label': 'high_value_payment'},
 {'score': 0.000718554831109941, 'label': 'direct_debit'},
 {'score': 0.0003914953558705747, 'label': 'latest_transactions'},
 {'score': 0.00033592403633520007, 'label': 'joint_account'},
 {'score': 0.00033554446417838335, 'label': 'balance'},
 {'score': 0.0003300652024336159, 'label': 'address'},
 {'score': 0.00014600456051994115, 'label': 'atm_limit'},
 {'score': 0.00014576790272258222, 'label': 'app_error'},
 {'score': 8.662768232170492e-05, 'label': 'cash_deposit'},
 {'score': 6.548867531819269e-05, 'label': 'business_loan'}]

In [29]:
#实际的标签
id2label = minds.features["intent_class"].int2str
id2label(sample1["intent_class"])

'pay_bill'

In [42]:
#第二个任务：automatic speech recognitian
from transformers import pipeline
asr = pipeline("automatic-speech-recognition"
)
asr(audio_array)

[transformers] No model was supplied, defaulted to facebook/wav2vec2-base-960h and revision 22aad52.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'text': 'I WOULD LIKE TO PAY MY ELECTRICITY BILL USING MY CARD CAN YOU PLEASE ASSIST'}

In [ ]:
#第三个任务：text to speech（tts）
tts=pipeline("text-to-speech",model="suno/bark")

text = "好烦啊"
output = tts(text)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=18) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
#观察输出
output

In [ ]:
#播放
from IPython.display import Audio

Audio(
    output["audio"],
    rate=output["sampling_rate"]
)